# Examen bimestral

**Introducción a los Sistemas de Información**

Prof. Iván Carrera

Ingeniería en Ciencia de Datos e Inteligencia Artificial · Escuela Politécnica Nacional

**Calificación total:** 10 puntos.

## Caso de estudio · Papelería universitaria

Una papelería universitaria registra las compras de sus sucursales Central, Norte y Sur en un sistema de punto de venta. Cada compra genera un ticket que puede incluir varios productos. La persona encargada de la papelería necesita resumir estos registros para conocer qué se vendió, cómo se pagaron las compras y cómo variaron las ventas entre sucursales y meses.

Elabore los reportes solicitados a partir de los datos.

In [ ]:
import os
import pandas as pd
from IPython.display import display

archivo_productos = 'productos.csv' if os.path.exists('productos.csv') else 'productos_.csv'
productos = pd.read_csv(archivo_productos)
clientes = pd.read_csv('clientes.csv')
ventas = pd.read_csv('ventas.csv')

ventas['importe_linea'] = ventas['cantidad'] * ventas['precio_unitario']
ventas_detalle = (
    ventas
    .merge(productos, on='id_producto', how='left')
    .merge(clientes, left_on='id_cliente', right_on='id', how='left')
)

### 1. Líneas, tickets y unidades

¿Cuántas líneas de venta, cuántos tickets distintos y cuántas unidades vendidas hay en el archivo? Explique qué representa cada conteo.

In [ ]:
reporte_1 = pd.DataFrame({
    'Métrica': ['Líneas de venta', 'Tickets distintos', 'Unidades vendidas'],
    'Conteo': [
        len(ventas),
        ventas['id_ticket'].nunique(),
        ventas['cantidad'].sum()
    ],
    'Qué representa cada conteo': [
        'Total de filas registradas en ventas.csv (cada producto incluido dentro de un ticket).',
        'Total de compras o transacciones únicas realizadas (valores únicos de id_ticket).',
        'Cantidad total de artículos vendidos en todas las líneas de venta (suma de cantidad).'
    ]
})

display(reporte_1.style.hide(axis='index').format({'Conteo': '{:,}'}))

### 2. Tickets por sucursal

¿Cuántos tickets se registraron en cada sucursal? ¿Cuál registró más?

In [ ]:
reporte_2 = (
    ventas.groupby('sucursal', as_index=False)['id_ticket']
    .nunique()
    .rename(columns={'sucursal': 'Sucursal', 'id_ticket': 'Tickets registrados'})
    .sort_values('Tickets registrados', ascending=False)
    .reset_index(drop=True)
)
max_tickets = reporte_2['Tickets registrados'].max()
reporte_2['Observación'] = reporte_2['Tickets registrados'].apply(
    lambda x: 'Sucursal con mayor cantidad de tickets' if x == max_tickets else ''
)

display(reporte_2.style.hide(axis='index').format({'Tickets registrados': '{:,}'}))

### 3. Métodos de pago

¿Cuántos tickets se pagaron con cada método y cuántos tienen ese dato sin registrar?

In [ ]:
tickets_pago = ventas[['id_ticket', 'metodo_pago']].drop_duplicates().copy()
tickets_pago['metodo_pago'] = tickets_pago['metodo_pago'].fillna('Sin registrar')

reporte_3 = (
    tickets_pago.groupby('metodo_pago', as_index=False)['id_ticket']
    .count()
    .rename(columns={'metodo_pago': 'Método de pago', 'id_ticket': 'Cantidad de tickets'})
    .sort_values('Cantidad de tickets', ascending=False)
    .reset_index(drop=True)
)

display(reporte_3.style.hide(axis='index').format({'Cantidad de tickets': '{:,}'}))

### 4. Precios de lista por categoría

¿Cuál es el precio de lista mínimo, máximo y promedio de los productos de cada categoría? Use el catálogo de productos.

In [ ]:
reporte_4 = (
    productos.groupby('categoria', as_index=False)['precio_lista']
    .agg(
        Precio_minimo='min',
        Precio_maximo='max',
        Precio_promedio='mean'
    )
    .rename(columns={
        'categoria': 'Categoría',
        'Precio_minimo': 'Precio mínimo',
        'Precio_maximo': 'Precio máximo',
        'Precio_promedio': 'Precio promedio'
    })
)

display(
    reporte_4.style.hide(axis='index').format({
        'Precio mínimo': '${:,.2f}',
        'Precio máximo': '${:,.2f}',
        'Precio promedio': '${:,.2f}'
    })
)

### 5. Unidades por sucursal y categoría

¿Cuántas unidades vendió cada sucursal de cada categoría? Presente una tabla cruzada con totales por fila y por columna. Compruebe que el total general coincide con la suma de `cantidad` en `ventas.csv`.

In [ ]:
reporte_5 = pd.crosstab(
    index=ventas_detalle['sucursal'],
    columns=ventas_detalle['categoria'],
    values=ventas_detalle['cantidad'],
    aggfunc='sum',
    margins=True,
    margins_name='Total'
)
reporte_5.index.name = 'Sucursal'
reporte_5.columns.name = 'Categoría'

verificacion_5 = pd.DataFrame({
    'Comprobación': [
        'Total general tabla cruzada',
        'Suma de cantidad en ventas.csv',
        '¿Coinciden ambos totales?'
    ],
    'Valor': [
        f"{reporte_5.loc['Total', 'Total']:,}",
        f"{ventas['cantidad'].sum():,}",
        'Sí' if reporte_5.loc['Total', 'Total'] == ventas['cantidad'].sum() else 'No'
    ]
})

display(reporte_5.style.format('{:,}'))
display(verificacion_5.style.hide(axis='index'))

### 6. Importe de ventas por sucursal

¿Cuál fue el importe total de ventas de cada sucursal? Calcule primero el precio total de cada línea como `cantidad × precio_unitario`.

In [ ]:
reporte_6 = (
    ventas.groupby('sucursal', as_index=False)['importe_linea']
    .sum()
    .rename(columns={'sucursal': 'Sucursal', 'importe_linea': 'Importe total'})
    .sort_values('Importe total', ascending=False)
    .reset_index(drop=True)
)

display(reporte_6.style.hide(axis='index').format({'Importe total': '${:,.2f}'}))

### 7. Tamaño de las compras

¿Cuántos tickets corresponden a compras de **hasta 7 unidades**, **de 8 a 14 unidades** y **de 15 o más unidades**? Sume primero las cantidades de todas las líneas de cada ticket.

In [ ]:
unidades_por_ticket = ventas.groupby('id_ticket', as_index=False)['cantidad'].sum()
rangos = ['Hasta 7 unidades', 'De 8 a 14 unidades', 'De 15 o más unidades']
unidades_por_ticket['Rango'] = pd.cut(
    unidades_por_ticket['cantidad'],
    bins=[0, 7, 14, float('inf')],
    labels=rangos
)

reporte_7 = (
    unidades_por_ticket['Rango']
    .value_counts()
    .reindex(rangos)
    .reset_index()
)
reporte_7.columns = ['Tamaño de la compra', 'Cantidad de tickets']

display(reporte_7.style.hide(axis='index').format({'Cantidad de tickets': '{:,}'}))

### 8. Ventas por mes

¿Cuántos tickets y qué importe total se registraron en cada mes? ¿En qué mes se alcanzó el mayor importe?

In [ ]:
ventas['mes'] = pd.to_datetime(ventas['fecha']).dt.to_period('M').astype(str)

reporte_8 = (
    ventas.groupby('mes', as_index=False)
    .agg(
        Tickets=('id_ticket', 'nunique'),
        Importe_total=('importe_linea', 'sum')
    )
    .rename(columns={'mes': 'Mes', 'Importe_total': 'Importe total'})
)
max_importe = reporte_8['Importe total'].max()
reporte_8['Observación'] = reporte_8['Importe total'].apply(
    lambda x: 'Mes con mayor importe' if x == max_importe else ''
)

display(
    reporte_8.style.hide(axis='index').format({
        'Tickets': '{:,}',
        'Importe total': '${:,.2f}'
    })
)

### 9. Producto más vendido por sucursal

¿Qué producto acumuló más unidades vendidas en cada sucursal? Indique su código, nombre y cantidad vendida. Si hay un empate, incluya todos los productos empatados.

In [ ]:
ventas_producto_sucursal = (
    ventas_detalle.groupby(['sucursal', 'id_producto', 'producto'], as_index=False)['cantidad']
    .sum()
)
max_por_sucursal = ventas_producto_sucursal.groupby('sucursal')['cantidad'].transform('max')

reporte_9 = (
    ventas_producto_sucursal[ventas_producto_sucursal['cantidad'] == max_por_sucursal]
    .sort_values(['sucursal', 'id_producto'])
    .reset_index(drop=True)
    .rename(columns={
        'sucursal': 'Sucursal',
        'id_producto': 'Código',
        'producto': 'Producto',
        'cantidad': 'Unidades vendidas'
    })
)

display(reporte_9.style.hide(axis='index').format({'Unidades vendidas': '{:,}'}))

### 10. Compras por tipo de cliente

Para cada tipo de cliente, ¿cuántos clientes distintos compraron, cuántos tickets generaron y cuál fue el importe total de sus compras?

In [ ]:
reporte_10 = (
    ventas_detalle.groupby('tipo_cliente', as_index=False)
    .agg(
        Clientes_distintos=('id_cliente', 'nunique'),
        Tickets_generados=('id_ticket', 'nunique'),
        Importe_total=('importe_linea', 'sum')
    )
    .rename(columns={
        'tipo_cliente': 'Tipo de cliente',
        'Clientes_distintos': 'Clientes distintos',
        'Tickets_generados': 'Tickets generados',
        'Importe_total': 'Importe total'
    })
    .sort_values('Importe total', ascending=False)
    .reset_index(drop=True)
)

display(
    reporte_10.style.hide(axis='index').format({
        'Clientes distintos': '{:,}',
        'Tickets generados': '{:,}',
        'Importe total': '${:,.2f}'
    })
)